In [82]:
import pandas as pd
import numpy as np
import os
from tqdm import tqdm

# Load the data
imdb_movies_df = pd.read_csv("dataset/IMDB_movies.csv", sep=",")
train_df = pd.read_csv('dataset/dataframe.csv')

In [70]:
# Check for null values
print(train_df.isnull().sum())

ID                     0
Movie_Name             0
Movie_Rating           0
Movie_Certificate    159
Movie_Year             0
Movie_Runtime          0
Movie_Cast             0
Movie_Genre            0
Movie_Poster           0
Movie_Description      0
Movie_Poster_HD        0
All_Movie_Info         0
dtype: int64


In [49]:
#A lot of columns have missing values, we will adapt our code accordingly

In [50]:
# Split the Genre column into a list of genres
imdb_movies_df['genre'] = imdb_movies_df['genre'].str.split(',')

# Encode genres into separate columns
imdb_movies_df = imdb_movies_df.join(imdb_movies_df.pop('genre').str.get_dummies(','))

In [71]:
from keras.applications.vgg16 import VGG16
from keras.preprocessing import image
from keras.applications.vgg16 import preprocess_input
import numpy as np
import tensorflow as tf
from tensorflow.keras.utils import load_img, img_to_array
import cv2
import dlib
from PIL import Image
from pytesseract import image_to_string


In [72]:
model = VGG16(weights='imagenet', include_top=False)

In [ ]:
for movie_id in train_df['ID']:
    image_path = os.path.join('dataset/poster/' + str(movie_id) + '.jpg')
    
    if os.path.exists(image_path):
        print(movie_id)
        img = cv2.imread(image_path)
        # Apply KPI extraction functions on the image
        # Append KPIs to the dataframe
        # ...
    else:
        print("Error on file " + 'dataset/Multi_Label_dataset/Images/' + movie_id + '.jpg')

In [78]:
def avg_rgb_intensity(image):
    average_color_per_row = np.average(image, axis=0)
    average_color = np.average(average_color_per_row, axis=0)
    return average_color

In [79]:
import dlib

def has_face_dlib(image):
    gray = cv2.cvtColor(image, cv2.COLOR_BGR2GRAY)
    detector = dlib.get_frontal_face_detector()
    faces = detector(gray)
    return len(faces) > 0


In [80]:
def num_faces_dlib(image):
    gray = cv2.cvtColor(image, cv2.COLOR_BGR2GRAY)
    detector = dlib.get_frontal_face_detector()
    faces = detector(gray)
    return len(faces)

In [84]:
for movie_id in tqdm(train_df['ID']):
    image_path = os.path.join('dataset/poster/' + str(movie_id) + '.jpg')
    
    if os.path.exists(image_path):
        img = cv2.imread(image_path)
        # Calculate KPIs
        avg_color = avg_rgb_intensity(img)
        has_face = has_face_dlib(img)
        num_faces = num_faces_dlib(img)

        # Append KPIs to the dataframe
        train_df.loc[train_df['ID'] == movie_id, 'avg_red'] = avg_color[0]
        train_df.loc[train_df['ID'] == movie_id, 'avg_green'] = avg_color[1]
        train_df.loc[train_df['ID'] == movie_id, 'avg_blue'] = avg_color[2]
        train_df.loc[train_df['ID'] == movie_id, 'has_face'] = has_face
        train_df.loc[train_df['ID'] == movie_id, 'num_faces'] = num_faces
    else:
        print("Error: file does not exist", 'dataset/poster/' + str(movie_id) + '.jpg')


100%|██████████| 1000/1000 [16:21<00:00,  1.02it/s]


In [98]:
def has_text_top_quarter(image):
    height, _, _ = image.shape
    top_quarter = image[:height//4, :]
    text = image_to_string(top_quarter)
    return 0 if text.strip() == "" else 1

In [99]:
for movie_id in tqdm(train_df['ID']):
    image_path = os.path.join('dataset/poster/' + str(movie_id) + '.jpg')
    
    if os.path.exists(image_path):
        img = cv2.imread(image_path)

        # Append KPIs to the dataframe
        train_df.loc[train_df['ID'] == movie_id, 'has_text_top_quarter'] = has_text_top_quarter(img)

    else:
        print("Error: file does not exist", 'dataset/poster/' + str(movie_id) + '.jpg')

 41%|████      | 409/1000 [03:36<04:15,  2.32it/s]

In [85]:
train_df.to_csv("test.csv")